# MediaPipe

### Función de dibujo

In [4]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)

# conexiones de pose sin tener en cuenta la muñeca y los dedos 
# (puntos 17, 18, 19, 20, 21, 22)
conexiones_poses = [
    (11, 12), (11, 13), (12, 14), (13,15), (14,16), # Brazos hasta codo
    (11, 23), (12, 24), (23, 24), # Tronco
    (23, 25), (24, 26), (25, 27), (26, 28), # Piernas
    (27, 29), (29, 31), (28, 30), (30, 32)  # Pies
]

puntos_excluidos = list(range(0,11)) + list(range(17,23))

def draw_complete_skeleton(rgb_image, resultado_pose, resultado_mano):
    annotated_image = np.copy(rgb_image)
    h, w, _ = annotated_image.shape
    #mano_izquierda, mano_derecha = False, False

    # dibujar las manos
    if resultado_mano and resultado_mano.hand_landmarks:
        hand_landmarks_list = resultado_mano.hand_landmarks
        handedness_list = resultado_mano.handedness
        #(mano_izquierda, mano_derecha) = detecta_mano(handedness_list)

        # iterar sobre las manos detectadas (1 o 2)
        for idx in range(len(hand_landmarks_list)):
            hand_landmarks = hand_landmarks_list[idx]
            handedness = handedness_list[idx]

            # dibujar los marcadores
            mp_drawing.draw_landmarks(
            annotated_image,
            hand_landmarks,
            mp_hands.HAND_CONNECTIONS,
            mp_drawing_styles.get_default_hand_landmarks_style(),
            mp_drawing_styles.get_default_hand_connections_style())

            # obtener la esquina superior izquierda del bounding box de la mano detectada
            height, width, _ = annotated_image.shape
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * width)
            text_y = int(min(y_coordinates) * height) - MARGIN

            # indicar si la mano es la derecha o la izquierda (left or right)
            cv2.putText(annotated_image, f"{handedness[0].category_name}",
                        (text_x, text_y), cv2.FONT_HERSHEY_DUPLEX,
                        FONT_SIZE, HANDEDNESS_TEXT_COLOR, FONT_THICKNESS, cv2.LINE_AA)


    # dibujar pose filtrada
    # si la mano de ese brazo es detectada, NO se dibuja la muñeca de la pose,
    # en caso de que la mano no se detecte, SÍ se dibuja la muñeca de ese brazo
    if resultado_pose and resultado_pose.pose_landmarks:
        # copia de las listas
        conexiones_poses_copia = list(conexiones_poses)
        puntos_excluidos_copia = list(puntos_excluidos)
        for landmarks in resultado_pose.pose_landmarks:
            # dibujar solo los puntos que no estén excluidos
            for idx, lm in enumerate(landmarks):
                if idx in puntos_excluidos_copia: 
                    continue
                cx, cy = int(lm.x * w), int(lm.y * h)
                cv2.circle(annotated_image, (cx, cy), 3, (0, 255, 0), -1)

            # dibujar las conexiones de pose
            for a, b in conexiones_poses_copia:
                p1, p2 = landmarks[a], landmarks[b]
                cv2.line(annotated_image, 
                         (int(p1.x * w), int(p1.y * h)), 
                         (int(p2.x * w), int(p2.y * h)), 
                         (255, 0, 0), 2)


    return annotated_image

### One euro

In [5]:
import math

def smoothing_factor(t_e, cutoff):
    r = 2 * math.pi * cutoff * t_e
    return r / (r + 1)


def exponential_smoothing(a, x, x_prev):
    return a * x + (1 - a) * x_prev


class OneEuroFilter:
    def __init__(self, t0, x0, dx0=0.0, min_cutoff=1.0, beta=0.0,d_cutoff=1.0):
        self.min_cutoff = float(min_cutoff)
        self.beta = float(beta)
        self.d_cutoff = float(d_cutoff)
        self.x_prev = float(x0)
        self.dx_prev = float(dx0)
        self.t_prev = float(t0)

    def __call__(self, t, x):
        t_e = t - self.t_prev

        if t_e <= 0: 
            return self.x_prev

        a_d = smoothing_factor(t_e, self.d_cutoff)
        dx = (x - self.x_prev) / t_e
        dx_hat = exponential_smoothing(a_d, dx, self.dx_prev)

        cutoff = self.min_cutoff + self.beta * abs(dx_hat)
        a = smoothing_factor(t_e, cutoff)
        x_hat = exponential_smoothing(a, x, self.x_prev)

        self.x_prev = x_hat
        self.dx_prev = dx_hat
        self.t_prev = t

        return x_hat

# Modelo

In [6]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import json
import os
import numpy as np


BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

model_path_poselandmarker = "../landmarkers/pose_landmarker_full.task"
model_path_handlandmarker = "../landmarkers/hand_landmarker.task"

options_pose = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_poselandmarker),
    running_mode=vision.RunningMode.VIDEO,
    min_pose_detection_confidence=0.6,
    min_pose_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

options_hand = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path_handlandmarker),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)

#indices_a_incluir = list(range(11, 17)) + list(range(23, 33))
indices_a_incluir = list(range(0,33))

# parámetros 1 euro filter
# min_cutoff: 0.01 - 0.1 (menor = más estable en reposo, pero más lento)
# beta: 0.5 - 5.0 (mayor = responde más rápido al movimiento veloz)
CFG_1EURO = {'min_cutoff': 0.1, 'beta': 5, 'd_cutoff': 1.0}

videos_folder = "../videos"
extensiones = {'avi', 'mov', 'mp4'}

if not os.path.exists("../videos_marcados"): os.makedirs("../videos_marcados")
if not os.path.exists("../animaciones_json"): os.makedirs("../animaciones_json")

with os.scandir(videos_folder) as videos:
    for video in videos:
        video_full_name = os.path.split(video)[1].split('.')
        if video.is_file() and video_full_name[1] in extensiones:
            video_name = video_full_name[0]
            output_video = f"../videos_marcados/{video_name}_oneeuro.mp4"

            cap = cv2.VideoCapture(video.path)
            fps = cap.get(cv2.CAP_PROP_FPS)
            w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

            fps = fps if fps > 0 else 30.0

            writer = cv2.VideoWriter(
                output_video,
                cv2.VideoWriter_fourcc(*"mp4v"),
                fps,
                (w, h)
            )

            animacion_esqueleto = []
            filtros_euro = {} # estado de los filtros

            # marcadores
            with PoseLandmarker.create_from_options(options_pose) as pose_detect, \
                 HandLandmarker.create_from_options(options_hand) as hand_detect:

                frame_idx = 0
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret:
                        break

                    frame_timestamp_ms = int((frame_idx / fps) * 1000)
                    t_seconds = frame_timestamp_ms / 1000.0
                    
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                    resultado_pose = pose_detect.detect_for_video(mp_image, frame_timestamp_ms)
                    resultado_mano = hand_detect.detect_for_video(mp_image, frame_timestamp_ms)

                    datos_frame = {
                        "Name": frame_idx + 1,
                        "frame": frame_idx,
                        "timestamp": frame_timestamp_ms,
                        "pose": [],
                        "hands": []
                    }

                    # manos
                    if resultado_mano.hand_landmarks:
                        for i, hand in enumerate(resultado_mano.hand_landmarks):
                            lado = resultado_mano.handedness[i][0].category_name
                            puntos_unreal = []
                            for idx_pt, l in enumerate(hand):
                                ejes = {'x': l.x, 'y': l.y, 'z': l.z}
                                res_filtrado = {}

                                for axis, val in ejes.items():
                                    key = f"hand_{lado}_{idx_pt}_{axis}"
                                    if key not in filtros_euro:
                                        filtros_euro[key] = OneEuroFilter(t_seconds, val, **CFG_1EURO)
                                        res_filtrado[axis] = val
                                    else:
                                        res_filtrado[axis] = filtros_euro[key](t_seconds, val)

                                puntos_unreal.append({"x": res_filtrado['x'], "y": res_filtrado['y'], "z": res_filtrado['z']})
                                
                                # actualizar valores del marcador con los valores suavizados
                                l.x = res_filtrado['x']
                                l.y = res_filtrado['y']
                                l.z = res_filtrado['z']

                            datos_frame["hands"].append({"lado": lado, "puntos": puntos_unreal})

                    # pose
                    if resultado_pose.pose_landmarks:
                        p = resultado_pose.pose_landmarks[0]
                        for idx in indices_a_incluir:
                            ejes = {'x': p[idx].x, 'y': p[idx].y, 'z': p[idx].z}
                            res_filtrado = {}

                            for axis, val in ejes.items():
                                key = f"pose_{idx}_{axis}"
                                if key not in filtros_euro:
                                    filtros_euro[key] = OneEuroFilter(t_seconds, val, **CFG_1EURO)
                                    res_filtrado[axis] = val
                                else:
                                    res_filtrado[axis] = filtros_euro[key](t_seconds, val)
                            
                            datos_frame["pose"].append({"x": res_filtrado['x'], "y": res_filtrado['y'], "z": res_filtrado['z']})

                            p[idx].x = res_filtrado['x']
                            p[idx].y = res_filtrado['y']
                            p[idx].z = res_filtrado['z']

                    animacion_esqueleto.append(datos_frame)

                    # dibujar resultados
                    annotated_frame = draw_complete_skeleton(frame_rgb, resultado_pose, resultado_mano)
                    final_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)
                    writer.write(final_frame)

                    frame_idx += 1

            cap.release()
            writer.release()

            # exportar resultados en JSON
            json_folder = f"../animaciones_json/{video_name}_oneeuro.json"
            with open(json_folder, "w") as f:
                json.dump(animacion_esqueleto, f, indent=4)

            print(f"Animación 1Euro de {video_name} guardada.")


I0000 00:00:1776275046.905664  190334 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275046.957575  190338 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275046.968579  190342 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275046.978040  190349 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275046.981736  190352 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275046.987915  190353 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación 1Euro de video guardada.


I0000 00:00:1776275048.040080  190393 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275048.087232  190396 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275048.093317  190398 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275048.096505  190409 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275048.098555  190411 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275048.101196  190416 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación 1Euro de b guardada.
Animación 1Euro de u guardada.


I0000 00:00:1776275048.241374  190444 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275048.290660  190446 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275048.296765  190446 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275048.305110  190459 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275048.307316  190461 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275048.310380  190464 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275048.438183  190489 gl

...

I0000 00:00:1776275055.393067  191443 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275055.439738  191444 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275055.445760  191444 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275055.451510  191458 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275055.454565  191460 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275055.456924  191464 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275055.562033  191487 gl

Animación 1Euro de n guardada.
Animación 1Euro de y guardada.


I0000 00:00:1776275056.206316  191580 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275056.252535  191581 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275056.258237  191586 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275056.261594  191595 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275056.263662  191597 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275056.266077  191601 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


Animación 1Euro de x guardada.
Animación 1Euro de o guardada.


I0000 00:00:1776275056.711469  191624 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275056.762672  191627 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275056.768685  191634 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
I0000 00:00:1776275056.771861  191639 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M4 Pro
W0000 00:00:1776275056.773900  191641 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1776275056.776535  191644 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
